# UWCT Tritonia tracking, phase 2 (D-043): E010 detections on every val frame

**What this does:** runs E010's registered `best.pt` (imgsz 640, conf 0.001) on all **7,728 frames** of the 3 v8
**val** days (20130524, 20130529, 20130602) and saves one prediction file per frame. These detections are the
input of the tracker (phase 3, local). **Detections only, no tracks and no scores here; no test split anywhere**
(CLAUDE.md rule 7; the script refuses a test sequence). Frames are 15 s apart: sparse-frame/timelapse analysis,
never "video tracking".

**How to run (one GPU is enough; ESTIMATE about 10-20 min in total):**
1. Settings: Accelerator **GPU** (T4), **Internet on**; Add-ons -> Secrets -> `UWCT_GITHUB_TOKEN` (ticked).
2. **Add Input** -> `anusidhu7/uwct-tritonia-tracking-val` (the frames) and `anusidhu7/uwct-e010-weights` (E010).
3. Cell 1: set `COMMIT` to the pushed commit that contains this notebook.
4. Run cells 1-4, then cell 5 with `RUN_SMOKE = True` (50 frames; must end with `wrote ...`), then cell 6 with
   `RUN_DETECT = True` (all frames), then cell 7, and **download `d043_val_detections.zip`** before closing.

In [ ]:
# 1. Settings and helpers -- set COMMIT; keep everything else.
import os, re, subprocess, sys, json, time

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/tritonia-tracking"   # branch that holds COMMIT
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: full sha, pushed
SEQ_DATASET, SEQ_NAME = "uwct-tritonia-tracking-val", "tritonia_tracking_val"
WEIGHTS_DATASET = "uwct-e010-weights"
E010_SHA256 = "48d72b79770bf8085bf70530eae4c71648a0b6c88c1e3580b90ca84e98909ca5"  # experiments/registry.csv
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"

WORK = "/tmp"                          # scratch, not saved
REPO_DIR = f"{WORK}/uwct"
W = "/kaggle/working"
LOGS = f"{W}/d043_logs"
os.makedirs(LOGS, exist_ok=True)

assert "KAGGLE_KERNEL_RUN_TYPE" in os.environ, "this notebook is for Kaggle (GPU)"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd, env=None):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret(name)


print("settings OK")

In [ ]:
# 2. GPU and the attached inputs (read-only under /kaggle/input; exactly one match each).
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU -- switch the accelerator to GPU")

seq_hits = sorted(d for d, subdirs, files in os.walk("/kaggle/input")
                  if os.path.basename(d) == SEQ_NAME and "images" in subdirs and "sequence.json" in files and SEQ_DATASET in d)
assert len(seq_hits) == 1, f"expected one {SEQ_NAME}/ of {SEQ_DATASET} under /kaggle/input, found {seq_hits} -- Add Input"
SEQ_DIR = seq_hits[0]
w_hits = sorted(os.path.join(d, "best.pt") for d, _, files in os.walk("/kaggle/input")
                if "best.pt" in files and d.endswith("E010/train/weights") and WEIGHTS_DATASET in d)
assert len(w_hits) == 1, f"expected one E010/train/weights/best.pt of {WEIGHTS_DATASET}, found {w_hits} -- Add Input"
E010_WEIGHTS = w_hits[0]
print("sequence:", SEQ_DIR, "|", len(os.listdir(os.path.join(SEQ_DIR, "images"))), "frames")
print("weights:", E010_WEIGHTS)

In [ ]:
# 3. Check out the exact commit (read-only token from Kaggle Secrets, as in colab_train_tritonia.ipynb cell 4:
#    the token is only in the environment of the one git process, never in a URL, file, .git/config or output).
import stat

REPO_URL = f"https://github.com/{REPO}.git"
ASKPASS = f"{WORK}/uwct_git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0", "UWCT_GIT_TOKEN": get_secret(TOKEN_SECRET_NAME)}
    try:
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env, capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd=WORK)
else:
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")
open(f"{LOGS}/code_version.txt", "w").write(f"{REPO_URL}\n{BRANCH}\n{COMMIT}\n")

In [ ]:
# 4. Pinned GPU requirements (ultralytics==8.4.155; torch/CUDA from Kaggle) and the session environment report.
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)")
sh(sys.executable, "scripts/env_report.py", "--out", f"{LOGS}/env_report_session.json")

In [ ]:
# 5. Smoke test (CLAUDE.md section 6): the first 50 frames, every check included. NOT a result.
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
SMOKE_OUT = f"{W}/d043_smoke_" + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
sh(sys.executable, "scripts/predict_tritonia_sequence.py", "--sequence", SEQ_DIR, "--weights", E010_WEIGHTS,
   "--weights-sha256", E010_SHA256, "--experiment", "E010", "--imgsz", "640", "--out", SMOKE_OUT, "--limit", "50")
print("SMOKE RUN PASSED")

In [ ]:
# 6. D-043 phase 2: E010 on all 7,728 val frames (conf 0.001, imgsz 640). Val only; the test split is never touched.
RUN_DETECT = False  # set to True deliberately, after cell 5 passed
assert RUN_DETECT, "Detection not enabled"
OUT = f"{W}/d043_val_detections"
assert not os.path.exists(OUT), f"{OUT} already exists -- download it first; never overwrite a finished run"
sh(sys.executable, "scripts/predict_tritonia_sequence.py", "--sequence", SEQ_DIR, "--weights", E010_WEIGHTS,
   "--weights-sha256", E010_SHA256, "--experiment", "E010", "--imgsz", "640", "--out", OUT)
rec = json.load(open(f"{OUT}/predictions.json"))
print(json.dumps({k: rec[k] for k in ("seconds", "summary_conf_0.001", "summary_conf_0.16", "git")}, indent=1))

In [ ]:
# 7. Download link: d043_val_detections.zip = predictions/ (one txt per frame), predictions.json, logs. Small.
import zipfile
from IPython.display import FileLink
os.chdir(W)
with zipfile.ZipFile("d043_val_detections.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("d043_val_detections", "d043_logs"):
        for d, _, files in os.walk(top):
            for n in sorted(files):
                z.write(os.path.join(d, n))
print("d043_val_detections.zip", os.path.getsize("d043_val_detections.zip"), "bytes")
display(FileLink("d043_val_detections.zip"))